# Compare two teams
Fit a model on all matches, rank the current teams and compare any two opponents. `strength` is a team's rating against an average unseen opponent with home advantage removed, in the model's own units (Elo points, or goals for the batch models). The chosen model is `elo_sqrt`.

In [1]:
%matplotlib inline
import sys, json, pathlib, warnings
warnings.filterwarnings("ignore")
# make the repo root importable no matter which folder the notebook is opened from
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "ratings").is_dir())
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from ratings import base
from ratings.data import load_matches
from ratings.tune import PARAMS_PATH, load_best
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from ratings.matchup import fit_final, rankings, predict_matchup, team_names

In [2]:
GENDER, MODEL = "male", "elo_sqrt"
model, mapper, matches = fit_final(GENDER, MODEL)
names = team_names()
ranks = rankings(model, matches, names)
print(f"{MODEL}, {GENDER}: {len(ranks)} teams played in {matches['year'].max()}")
display(ranks.head(15).round(1))
display(ranks.tail(5).round(1))

elo_sqrt, male: 265 teams played in 2026


,team_id,strength,team
0,27948,554.0,Georgetown Hoyas
1,28651,517.4,SMU Mustangs
2,27990,411.7,Washington Huskies
3,28672,409.8,Clemson Tigers
4,26951,387.1,Portland Pilots
5,28448,362.1,UMKC Kangaroos
6,27951,358.2,Indiana Hoosiers
7,24425,358.1,UCLA Bruins
8,24498,342.8,Akron Zips
9,26948,342.3,North Carolina Heels


,team_id,strength,team
260,28090,-322.9,NKU Norse
261,28222,-339.7,Canisius Golden Griffins
262,24698,-372.4,Presbyterian BlueHose
263,27950,-372.7,Howard Bison
264,67915,-545.3,Chicago State Cougars


## Compare two teams by name
Change `HOME` and `AWAY` to any team names (see `names` for the exact spelling). A neutral venue averages the gap over both home advantages.

In [3]:
HOME, AWAY = ranks.iloc[0]["team"], ranks.iloc[len(ranks) // 2]["team"]
ids = {name: team_id for team_id, name in names.items()}
for neutral in (False, True):
    out = predict_matchup(model, mapper, ids[HOME], ids[AWAY], neutral=neutral)
    print(f"{HOME} vs {AWAY}", "(neutral site)" if neutral else "(home)")
    print({k: round(float(v), 3) for k, v in out.items()})

Georgetown Hoyas vs St. Thomas Tommies (home)
{'gap': 635.519, 'p_home': 0.913, 'p_draw': 0.054, 'p_away': 0.033, 'exp_goal_diff': 3.003}
Georgetown Hoyas vs St. Thomas Tommies (neutral site)
{'gap': 565.519, 'p_home': 0.885, 'p_draw': 0.071, 'p_away': 0.044, 'exp_goal_diff': 2.684}


## Different models, same match

In [4]:
rows = []
for name in ["elo_none", "elo_sqrt", "elo_xg", "glicko", "pi", "dixon_coles_xg"]:
    m, mp, _ = fit_final(GENDER, name, matches)
    rows.append({"model": name, **predict_matchup(m, mp, ids[HOME], ids[AWAY])})
display(pd.DataFrame(rows).set_index("model").round(3))

,gap,p_home,p_draw,p_away,exp_goal_diff
model,,,,,
elo_none,553.640,0.890,0.068,0.042,2.727
elo_sqrt,635.519,0.913,0.054,0.033,3.003
elo_xg,539.534,0.927,0.046,0.027,3.201
glicko,508.598,0.869,0.080,0.051,2.539
pi,601.700,0.954,0.029,0.017,3.727
dixon_coles_xg,2.825,0.903,0.061,0.036,2.821
